In [1]:
import os
import json
import glob

import matplotlib.pyplot as plt

import pandas as pd
from scipy.stats import wilcoxon

from recsysconfident.utils.files import sort_paths_by_datetime

def find_subfolders_with_prefix(root_folder: str, prefix: str):

  subfolders = []
  for dirpath, dirnames, filenames in os.walk(root_folder):
    for dirname in dirnames:
      if dirname.startswith(prefix):
        subfolders.append(os.path.join(dirpath, dirname))
  return subfolders

def read_json(path: str) -> dict:

    with open(path, 'r') as f:
        return json.load(f)

def generate_latex_table_from_dataframe(df: pd.DataFrame, caption: str, label: str, columns: list, bold: bool=False):
    """
    columns: models
    rows: metrics
    """

    df_without_std = df[~df.index.str.contains("std")]
    std_columns = []
    metrics_cols = []
    for col in list(df_without_std.columns):
        if "std" in col:
            std_columns.append(col)
        else:
            metrics_cols.append(col)

    if not columns:
        columns = metrics_cols

    for col in columns + std_columns:
        df_without_std.loc[:, col] = df_without_std[col].astype(float).round(3)

    df_bolded = df_without_std.astype(str)
    if bold:
        for idx, row in df_without_std[columns].iterrows():

            if "RMSE" in idx or "MAE" in idx:
                bold_value = row.min()
            else:
                bold_value = row.max()

            for col in columns:
                if row[col] == bold_value:
                    df_bolded.at[idx, col] = "\\textbf{"+str(row[col])+"}"

    for std_col in std_columns:
        col_name = std_col[:-4]
        formatted_col = df_bolded[col_name].astype(str) + "$\\pm$" + df_bolded[std_col].astype(str)
        df_bolded[col_name] = formatted_col

    df_bolded = df_bolded[columns]
    df_bolded = df_bolded.reset_index().rename(columns={'index': 'Metric'})

    df_res = df_bolded.T
    df_res.columns = df_res.iloc[0]
    df_res = df_res.drop(df_res.index[0])
    df_res = df_res.reset_index().rename(columns={'index': "Models"})

    latex_code = df_res.to_latex(
        label=label,
        caption=caption,
        index=False,
        escape=False,  # Prevent escaping special characters
        column_format="c" * len(df_without_std.columns)  # Center align columns
    )
    return latex_code

def get_models_metrics(dataset_uris, split_name, metrics) -> pd.DataFrame:

    models_metrics_dfs = {}
    for path in dataset_uris:
        if "data_splits" in path:
            continue

        setup = read_json(sort_paths_by_datetime(glob.glob(f"{path}/setup-*.json"))[-1])
        model_name = setup['model_name']

        metrics_list = sort_paths_by_datetime(glob.glob(f"{path}/metrics-*.json"))
        metrics_df = pd.DataFrame.from_dict([read_json(metrics_list[-1])[split_name]])
        metrics_df.columns = metrics_df.columns.str.upper()

        if model_name in models_metrics_dfs:
            models_metrics_dfs[model_name] = pd.concat([models_metrics_dfs[model_name], metrics_df[metrics]], axis=0)
        else:
            models_metrics_dfs[model_name] = metrics_df[metrics]

    return models_metrics_dfs

def plot_model_performance(data, metrics, model_names):
    markers = ['o', 's', '^', 'D', 'v', 'p', '*', 'h', 'X', 'd']

    for dataset, df in data.items():
        plt.figure(figsize=(10, 6))

        for metric in df.index:
            if metric in metrics:
                y_values = df.loc[metric, model_names]
                marker_style = markers[metrics.index(metric) % len(markers)]
                plt.scatter(model_names, y_values, label=metric, marker=marker_style)

        plt.title(f"Dataset: {dataset}")
        plt.xlabel('Models')
        plt.ylabel('Performance')
        plt.xticks(rotation=45)
        plt.legend(title="Metrics", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

In [4]:
metrics = ["RMSE", "MNDCG@10", "MAP@10", "MNDCG@3", "MAP@3"]

group_name = "proposal"
model_names = ['mf', 'dmf', 'dgat', 'deepergat2', 'deepergat3']
#group_name = "distribution_based"
#model_names = ['ordrec', 'cpmf', 'cbpmf', 'lbd']

split_name = "test"

datasets_uris = {
  "amazon-movies-tvs": find_subfolders_with_prefix(f"../runs/{group_name}/", "amazon-movies-tvs"),
  "ml-1m": find_subfolders_with_prefix(f"../runs/{group_name}/", "ml-1m"),
  #"netflix-prize": find_subfolders_with_prefix(f"../runs/{group_name}/", "netflix-prize"),
  "jester-joke": find_subfolders_with_prefix(f"../runs/{group_name}/", "jester-joke"),
}

In [5]:

alpha = 0.05
metrics_ds = {}

for dataset_name in datasets_uris.keys():

    models_metrics_dfs_dict = get_models_metrics(
        datasets_uris[dataset_name], "test", metrics
    )

    for model_name in model_names:

        mean_metrics_df = (
            models_metrics_dfs_dict[model_name]
            .astype(float)
            .mean()
            .to_frame(name=model_name)
        )

        std_metrics_df = (
            models_metrics_dfs_dict[model_name]
            .astype(float)
            .std()
            .to_frame(name=f"{model_name}_std")
        )

        if dataset_name in metrics_ds:
            metrics_ds[dataset_name] = pd.concat(
                [metrics_ds[dataset_name], mean_metrics_df, std_metrics_df],
                axis=1
            )
        else:
            metrics_ds[dataset_name] = pd.concat(
                [mean_metrics_df, std_metrics_df],
                axis=1
            )

    # =========================
    # 2. Wilcoxon: best vs second-best (per metric)
    # =========================
    wilcoxon_results = {}

    for metric in metrics:

        # compute mean per model for ranking
        means = {
            model_name: models_metrics_dfs_dict[model_name][metric]
            .astype(float)
            .mean()
            for model_name in model_names
        }

        # select best and second-best models
        best_model, second_model = sorted(
            means, key=means.get, reverse=True
        )[:2]

        x = models_metrics_dfs_dict[best_model][metric].astype(float).values
        y = models_metrics_dfs_dict[second_model][metric].astype(float).values

        # handle degenerate case (identical samples)
        if (x == y).all():
            stat = 0.0
            pval = 1.0
        else:
            stat, pval = wilcoxon(x, y, alternative="greater")

        wilcoxon_results[metric] = {
            "best_model": best_model,
            "second_model": second_model,
            "wilcoxon_stat": stat,
            "p": pval,
            "best_significantly_better": pval < alpha,
        }

    wilcoxon_df = pd.DataFrame(wilcoxon_results).T

    # =========================
    # 3. Attach Wilcoxon results
    # =========================
    metrics_ds[dataset_name] = metrics_ds[dataset_name].join(wilcoxon_df)


In [6]:
print(generate_latex_table_from_dataframe(metrics_ds['amazon-movies-tvs'],
                                          "Metrics of the proposed and baseline models in the test split of Amazon Movies TVs.",
                                          "tab:amazon-movies-tvs-performancess", model_names+["p"]).replace(" & ", "&"))

print(generate_latex_table_from_dataframe(metrics_ds['jester-joke'], 
                                          "Metrics of the proposed and baseline models' performances in the test split of the Jester Joke dataset.",
                                          "tab:jester-joke-performances", model_names+["p"]).replace(" & ", "&"))

print(generate_latex_table_from_dataframe(metrics_ds['ml-1m'], 
                                          "Proposed and baseline models' performance over the test split of the Movie Lens 1M dataset.",
                                            "tab:ml-1m-performances", model_names+["p"]).replace(" & ", "&"))


\begin{table}
\caption{Metrics of the proposed and baseline models in the test split of Amazon Movies TVs.}
\label{tab:amazon-movies-tvs-performancess}
\begin{tabular}{ccccccccccccccc}
\toprule
Models&RMSE&MNDCG@10&MAP@10&MNDCG@3&MAP@3 \\
\midrule
mf&0.967$\pm$0.011&0.299$\pm$0.015&0.265$\pm$0.014&0.345$\pm$0.018&0.337$\pm$0.018 \\
dmf&1.493$\pm$0.015&0.176$\pm$0.001&0.085$\pm$0.0&0.167$\pm$0.001&0.264$\pm$0.001 \\
dgat&0.95$\pm$0.008&0.144$\pm$0.021&0.149$\pm$0.016&0.112$\pm$0.03&0.117$\pm$0.03 \\
deepergat2&0.947$\pm$0.007&0.176$\pm$0.018&0.171$\pm$0.014&0.169$\pm$0.023&0.173$\pm$0.022 \\
deepergat3&0.948$\pm$0.007&0.175$\pm$0.019&0.169$\pm$0.015&0.17$\pm$0.025&0.172$\pm$0.023 \\
p&0.031&0.031&0.031&0.031&0.031 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}
\caption{Metrics of the proposed and baseline models' performances in the test split of the Jester Joke dataset.}
\label{tab:jester-joke-performances}
\begin{tabular}{ccccccccccccccc}
\toprule
Models&RMSE&MNDCG@10&MAP@10&